# FinDelta — Colab 一周实验

用云端 GPU 运行财报推理的 Base → LoRA SFT → DPO，4070 主机不需要联网。

**首次只运行第 1–5 步，确认 GPU 和环境。不要直接“全部运行”。** 后续训练单元有开关，默认关闭。

准备本对话提供的 **FinDelta-Colab-Week1.zip**。项目含公开财报数据，不需要上传简历、公司文件或账号凭证。

菜单：**运行时 → 更改运行时类型 → GPU**。可先试 T4；有 L4 也可。实际可用硬件由 Colab 决定。0.6B 是起点，GPU 显存和效果尚未在你的 Colab 账户实测。

官方说明：[资源与运行时限制](https://research.google.com/colaboratory/faq.html) · [付费方案](https://colab.research.google.com/signup)

输出写入你授权挂载的 Google Drive。训练中途被终止时，当前训练仍可能需要重跑；本版本未实现中途 checkpoint 恢复。不要假设付费运行时不会中断。


## 1. 检查云端 GPU

In [ ]:
import subprocess, sys, os, json, pathlib, shutil, zipfile, datetime
check = subprocess.run(["nvidia-smi"], text=True, capture_output=True)
print(check.stdout or check.stderr)
if check.returncode:
    raise RuntimeError("请先在运行时设置里选择 GPU，再运行本单元。")


## 2. 上传项目压缩包
上传本对话中的 FinDelta-Colab-Week1.zip（约 12MB）。代码和数据解压在云端本地磁盘，后续训练输出另存 Drive。

In [ ]:
from google.colab import files
PROJECT = pathlib.Path("/content/fin-delta")
if PROJECT.exists():
    print("项目已存在，保留现有文件：", PROJECT)
else:
    uploaded = files.upload()
    matches = [name for name in uploaded if name.endswith(".zip")]
    if len(matches) != 1:
        raise ValueError("请只上传一个 FinDelta 项目 zip")
    archive = pathlib.Path("/content") / pathlib.Path(matches[0]).name
    archive.write_bytes(uploaded[matches[0]])
    with zipfile.ZipFile(archive) as z:
        if "fin-delta/pyproject.toml" not in z.namelist():
            raise ValueError("这不是 FinDelta 项目包")
        for member in z.infolist():
            destination = (pathlib.Path("/content") / member.filename).resolve()
            if not destination.is_relative_to(PROJECT.resolve()):
                raise ValueError("压缩包包含项目目录之外的路径")
        z.extractall("/content")
    del uploaded
    print("已解压：", PROJECT)


## 3. 安装隔离环境
使用独立 Python 3.12 环境，避免更改 Colab Notebook 自身预装的 PyTorch。不需要重启 Notebook 内核。首次会下载依赖和 CUDA 组件。

In [ ]:
VENV = pathlib.Path("/content/findelta-venv")
PY = str(VENV / "bin/python")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.8.22"], check=True)
if not VENV.exists():
    subprocess.run([sys.executable, "-m", "uv", "venv", "--python", "3.12", "--managed-python", str(VENV)], check=True)
subprocess.run([sys.executable, "-m", "uv", "pip", "install", "--python", PY,
                "torch==2.8.0", "--index-url", "https://download.pytorch.org/whl/cu126"], check=True)
subprocess.run([sys.executable, "-m", "uv", "pip", "install", "--python", PY,
                "-e", str(PROJECT) + "[train,cuda]"], check=True)
ENV = dict(os.environ, HF_HOME="/content/findelta-hf-cache", TOKENIZERS_PARALLELISM="false")
subprocess.run([PY, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=PROJECT, env=ENV, check=True)


## 4. 挂载 Google Drive，固定本次实验目录
需你在 Google 弹窗中授权。Notebook 仅在 MyDrive/FinDelta 下写项目输出。断线重连后填回同一个 RUN_ID，可以继续使用已完成的结果；切勿覆盖旧实验。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
RUN_ID = "week1-001" #@param {type:"string"}
if not RUN_ID or any(c not in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-_" for c in RUN_ID):
    raise ValueError("RUN_ID 只使用英文、数字、-、_。新实验换一个 ID。")
RUN = pathlib.Path("/content/drive/MyDrive/FinDelta") / RUN_ID
RUN.mkdir(parents=True, exist_ok=True)
def fd(*args):
    command = [PY, "-m", "findelta", *map(str, args)]
    print("运行:", " ".join(command), flush=True)
    subprocess.run(command, cwd=PROJECT, env=ENV, check=True)
def predict_once(name, data, adapter=None, samples=1, temperature=0):
    output = RUN / (name + "-predictions.jsonl")
    if output.exists():
        if pathlib.Path(str(output) + ".manifest.json").exists():
            print("已完成，保留结果:", output)
            return output
        raise RuntimeError(f"检测到未完成预测 {output}；保留文件供排查，换一个 name 再运行。")
    args = ["predict", "--data", data, "--out", output, "--samples", samples, "--temperature", temperature]
    if adapter is not None:
        args += ["--adapter", adapter]
    fd(*args)
    return output
def train_once(name, mode, data, adapter=None, max_steps=None):
    output = RUN / name
    if (output / "run_manifest.json").exists():
        print("已完成，保留训练:", output)
        return output
    if output.exists() and any(output.iterdir()):
        raise RuntimeError(f"检测到未完成训练 {output}；当前版不恢复中途训练。保留后换 name 重跑。")
    cfg = "configs/4070-sft.json" if mode == "sft" else "configs/4070-dpo.json"
    args = ["train", "--mode", mode, "--config", cfg, "--data", data, "--out", output]
    if adapter is not None:
        args += ["--sft-adapter", adapter]
    if max_steps is not None:
        args += ["--max-steps", max_steps]
    fd(*args)
    return output
print("实验输出持久保存到:", RUN)


## 5. 保存硬件与数据记录
运行后把 hardware.json 的内容发回 Codex。至此可以停下；下面才开始模型下载与实验。

In [ ]:
report = subprocess.check_output([PY, "-m", "findelta", "doctor"], cwd=PROJECT, env=ENV, text=True)
print(report)
(RUN / "hardware.json").write_text(report)
hardware = json.loads(report)
if not hardware.get("cuda_available"):
    raise RuntimeError("独立环境未识别 CUDA，请把上面的报告和 nvidia-smi 输出发回排查。")
for source, target in [(PROJECT / "model-lock.json", RUN / "model-lock.json"),
                       (PROJECT / "data/cohort/v1/manifest.json", RUN / "cohort-manifest.json")]:
    if target.exists() and target.read_bytes() != source.read_bytes():
        raise RuntimeError("RUN_ID 对应的模型或数据已经变化，请使用新的 RUN_ID。")
    shutil.copyfile(source, target)
fd("export-sft")


## 6. 原模型基线（第 1 天）
勾选后运行。真实模型首次需要联网下载；200 道 dev 题，固定 greedy 输出预算。

In [ ]:
RUN_BASELINE = False #@param {type:"boolean"}
if RUN_BASELINE:
    pred = predict_once("base-dev", "data/cohort/v1/dev.jsonl")
    fd("evaluate", "--data", "data/cohort/v1/dev.jsonl", "--predictions", pred, "--out", RUN / "base-dev")
else:
    print("未启动基线；准备好后勾选 RUN_BASELINE。")


## 7. SFT 十步试跑（第 2 天）
先检查能否运行、loss 和峰值显存。默认配置适配 BF16，T4 等不支持时自动用 FP16。发生 OOM 时先发日志，不直接改评测子集。

In [ ]:
RUN_SFT_SMOKE = False #@param {type:"boolean"}
if RUN_SFT_SMOKE:
    train_once("sft-smoke", "sft", "data/cohort/v1/sft.jsonl", max_steps=10)


## 8. 正式 SFT + dev 对照（第 2–3 天）
从原始模型重新训练一个 epoch，不接续试跑。

In [ ]:
RUN_SFT = False #@param {type:"boolean"}
if RUN_SFT:
    adapter = train_once("sft", "sft", "data/cohort/v1/sft.jsonl")
    pred = predict_once("sft-dev", "data/cohort/v1/dev.jsonl", adapter)
    fd("evaluate", "--data", "data/cohort/v1/dev.jsonl", "--predictions", pred, "--out", RUN / "sft-dev")
    fd("compare", "--run", f"Base={RUN / 'base-dev'}", "--run", f"SFT={RUN / 'sft-dev'}", "--out", RUN / "dev-comparison.md")
    print((RUN / "dev-comparison.md").read_text())


## 9. 从训练集真实错误构造偏好（第 4 天）
前 100 道训练题，每题采样两次。候选偏好只来自训练集；可能少于 100 对。

In [ ]:
RUN_PREFERENCE_SAMPLING = False #@param {type:"boolean"}
if RUN_PREFERENCE_SAMPLING:
    sys.path.insert(0, str(PROJECT))
    from findelta.common import read_jsonl, write_jsonl
    sampling_data = PROJECT / "data/cohort/v1/train-sampling.jsonl"
    write_jsonl(sampling_data, read_jsonl(PROJECT / "data/cohort/v1/train.jsonl")[:100])
    pred = predict_once("train-samples", sampling_data, RUN / "sft", samples=2, temperature=.7)
    pairs = RUN / "preferences/model-v1"
    if not pairs.exists():
        fd("preferences", "--data", sampling_data, "--predictions", pred, "--out", pairs, "--limit", 100)
    print("审核说明：", pairs / "review.md")
    files.download(str(pairs / "review.md"))
    files.download(str(pairs / "review.csv"))


## 10. 人工审核后上传 review.csv
先读 review.md，检查 chosen/rejected 的运算、单位、期间和证据。在 CSV 中逐行填写 approve/reject。保持 ID 不变；没有把握就 reject。不要只凭数值验证批量 approve。

In [ ]:
UPLOAD_REVIEW = False #@param {type:"boolean"}
if UPLOAD_REVIEW:
    uploaded_review = files.upload()
    csv_names = [n for n in uploaded_review if n.lower().endswith(".csv")]
    if len(csv_names) != 1:
        raise ValueError("请只上传一份审核后的 CSV")
    reviewed = RUN / "preferences/model-v1/review-reviewed.csv"
    if reviewed.exists():
        raise ValueError("已有审核文件，请保留原文件并另建版本。")
    reviewed.write_bytes(uploaded_review[csv_names[0]])
    fd("export-dpo", "--candidates", RUN / "preferences/model-v1/candidates.jsonl",
       "--review", reviewed, "--out", RUN / "preferences/model-v1/approved.jsonl")


## 11. DPO + dev 评测（第 5 天）
合格偏好太少就先交付 Base/SFT，不必凑出 DPO 数字。DPO 参考模型为同一个 SFT checkpoint。

In [ ]:
RUN_DPO = False #@param {type:"boolean"}
if RUN_DPO:
    approved = RUN / "preferences/model-v1/approved.jsonl"
    if not approved.exists():
        raise ValueError("请先完成偏好审核与导出。")
    train_once("dpo-smoke", "dpo", approved, RUN / "sft", max_steps=10)
    train_once("dpo", "dpo", approved, RUN / "sft")
    pred = predict_once("dpo-dev", "data/cohort/v1/dev.jsonl", RUN / "dpo/policy")
    fd("evaluate", "--data", "data/cohort/v1/dev.jsonl", "--predictions", pred, "--out", RUN / "dpo-dev")
    fd("compare", "--run", f"Base={RUN / 'base-dev'}", "--run", f"SFT={RUN / 'sft-dev'}",
       "--run", f"DPO={RUN / 'dpo-dev'}", "--out", RUN / "dev-comparison.md")


## 12. 最终 test（第 6 天）
只在 dev 上固定方法后执行。不要根据 test 结果反复调参。DPO 可选；使用同一测试 cohort。

In [ ]:
RUN_FINAL_TEST = False #@param {type:"boolean"}
INCLUDE_DPO = False #@param {type:"boolean"}
if RUN_FINAL_TEST:
    models_to_test = [("Base", "base-test", None), ("SFT", "sft-test", RUN / "sft")]
    if INCLUDE_DPO:
        models_to_test.append(("DPO", "dpo-test", RUN / "dpo/policy"))
    compare_args = ["compare"]
    for label, name, adapter in models_to_test:
        pred = predict_once(name, "data/cohort/v1/test.jsonl", adapter)
        fd("evaluate", "--data", "data/cohort/v1/test.jsonl", "--predictions", pred, "--out", RUN / name)
        compare_args += ["--run", f"{label}={RUN / name}"]
    fd(*compare_args, "--out", RUN / "test-comparison.md")
    print((RUN / "test-comparison.md").read_text())


## 13. 下载结果并结束 GPU 会话
结束时从菜单选择“断开连接并删除运行时”，不要让付费 GPU 空闲。Google Drive 中已完成的结果会保留。下方只打包本次实验结果，不打包模型下载缓存。

In [ ]:
DOWNLOAD_RESULTS = False #@param {type:"boolean"}
if DOWNLOAD_RESULTS:
    result_zip = shutil.make_archive("/content/findelta-results-" + RUN_ID, "zip", RUN)
    files.download(result_zip)
print("结果目录:", RUN)
print("不再训练时，请手动断开连接并删除运行时。")


## 出错时发回什么

- 第 5 步 hardware.json 与报错。
- SFT/DPO 的 run_manifest.json，以及 dev 的 metrics.json。
- 无需发模型缓存、凭证、简历或公司材料。

此 Notebook 在本地完成 JSON/代码单元检查；核心训练代码已用微型 CPU 模型集成验证。**尚未在 Colab GPU 上端到端执行**，具体硬件、驱动与额度以你的会话实测为准。
